In [10]:
import os
import sys
from IPython.display import Markdown, display, update_display
from openai import OpenAI
import json

from pdf_text import extract_pdf_text

In [11]:
openai = OpenAI(
    base_url="http://localhost:11500/v1",
    api_key="ollama"
)

MODEL = "llama3.2:3b"

## Extract text from resume

In [12]:
extract_pdf_text("/home/jaimin.rana@agshealth.com/Documents/resumes/Jaimin_resume.pdf")

'Jaimin Rana\nAhmedabad, Gujarat, India\njaiminrana1102@gmail.com\n+919510425401\nin/jaimin-rana-56a756212\ngithub.com/J4Jaimin\nSUMMARY\nExperienced Software Developer with 2 years of expertise in building scalable backend systems using Node.js, Express, and MongoDB. Strong understanding of \nRESTful APIs, authentication, and database management. Quick learner with basic knowledge of React and a flexible approach to new technologies.\nEXPERIENCE\nSoftware Engineer\nAGS Health Pvt. Ltd.\nAugust 2023 - Present Ahmedabad, Gujarat, India\n,\nDeveloped and maintained scalable backend systems using Node.js and Express.js, improving API response times by up to 30%.\n•\nDesigned and optimized MongoDB and MySQL databases, resulting in 25% faster query performance and improved data integrity.\n•\nCollaborated on full-stack applications with React.js, HTML, and CSS, enhancing UI responsiveness and cross-browser compatibility.\n•\nContainerized applications using Docker and implemented CI/CD pipe

### Convert raw resume text into structured JSON.

In [21]:
def convert_raw_text_to_json(raw_text):
    response = openai.chat.completions.create(
        model=MODEL,
        response_format={"type": "json_object"},
        messages=[
            {
                "role": "system",
                "content": "Extract structured resume information in JSON format."
            },
            {
                "role": "user",
                "content": raw_text
            }
        ]
    )

    return json.loads(response.choices[0].message.content)

In [22]:
candidate_info = convert_raw_text_to_json(extract_pdf_text("/home/jaimin.rana@agshealth.com/Documents/resumes/Jaimin_resume.pdf"))

In [23]:
candidate_info

{'name': 'Jaimin Rana',
 'email': 'jaiminrana1102@gmail.com',
 'phone': '+919510425401',
 'location': {'city': 'Ahmedabad', 'state': 'Gujarat', 'country': 'India'},
 'github': 'https://github.com/J4Jaimin',
 'summary': 'Experienced Software Developer with 2 years of expertise in building scalable backend systems using Node.js, Express, and MongoDB. Strong understanding of ÖRESTful APIs, authentication, and database management. Quick learner with basic knowledge of React and a flexible approach to new technologies.×',
 'experience': [{'title': 'Software Engineer',
   'company': 'AGS Health Pvt. Ltd.',
   'joinDate': 'August 2023',
   'location': {'city': 'Ahmedabad', 'state': 'Gujarat', 'country': 'India'},
   'summary': 'Developed and maintained scalable backend systems using Node.js and Express.js, improving API response times by up to 30%.×\nDesigned and optimized MongoDB and MySQL databases, resulting in 25% faster query performance and improved data integrity.×\nCollaborated on ful

In [24]:
job_description = """
    Job Title: Backend Developer (Node.js) – 2.5+ Years Experience

    Location: Remote / Hybrid
    Experience Required: 2.5+ Years
    Employment Type: Full-Time

    About the Role

    We are looking for a skilled Backend Developer with strong experience in Node.js and modern backend development practices. The ideal candidate should have hands-on experience building scalable REST APIs, working with databases, integrating third-party services, and deploying applications in cloud environments. Basic frontend knowledge in React.js or Next.js and intermediate DevOps exposure are highly preferred.

    Key Responsibilities

    - Design, develop, and maintain scalable backend services using Node.js and Express.js
    - Build secure and high-performance RESTful APIs and microservices
    - Work with databases such as MongoDB, MySQL, or PostgreSQL
    - Implement authentication and authorization mechanisms (JWT, OAuth, RBAC)
    - Optimize application performance, scalability, and reliability
    - Collaborate with frontend developers to integrate APIs with React.js or Next.js applications
    - Integrate third-party APIs, payment gateways, and cloud services
    - Participate in code reviews, debugging, and production issue resolution
    - Write reusable, testable, and maintainable code
    - Work with Docker containers and CI/CD pipelines for deployment automation
    - Monitor application logs, server performance, and cloud infrastructure

    Required Skills

    - 2.5+ years of experience in backend development
    - Strong proficiency in Node.js and Express.js
    - Good understanding of JavaScript and asynchronous programming
    - Experience with REST APIs and backend architecture
    - Hands-on experience with MongoDB and/or SQL databases
    - Knowledge of authentication systems such as JWT and OAuth
    - Familiarity with Git and collaborative development workflows
    - Understanding of caching, queues, and background jobs
    - Experience with Docker and containerized applications
    - Intermediate knowledge of cloud platforms such as AWS or Azure
    - Basic understanding of Linux servers and deployment processes
    - Familiarity with frontend technologies like React.js or Next.js

    Preferred Skills

    - Experience with TypeScript
    - Knowledge of Redis, RabbitMQ, or Kafka
    - Understanding of system design and scalable architecture
    - Experience with serverless services or cloud-native deployments
    - Familiarity with Kubernetes and infrastructure automation
    - Exposure to monitoring/logging tools such as Prometheus, Grafana, or ELK stack
    - Understanding of secure coding practices and API security

    Nice to Have

    - Experience working in Agile/Scrum teams
    - Knowledge of AI/LLM integrations or automation workflows
    - Open-source contributions or personal backend projects
    - Experience building SaaS products or internal developer platforms

    What We Offer

    - Opportunity to work on scalable and modern backend systems
    - Exposure to cloud infrastructure and DevOps workflows
    - Collaborative engineering culture
    - Flexible work environment
    - Career growth and learning opportunities
"""

In [41]:
system_prompt = """
You are an expert technical recruiter, ATS evaluator, and career strategist.

Your task is to analyze how well a candidate matches a job description and generate a professional markdown report.

The report should include:
- Match summary
- Candidate strengths
- Skill gaps
- ATS optimization suggestions
- Resume enhancement ideas
- Interview preparation guidance
- Realistic match score with reasoning

Rules:
- Be realistic and recruiter-focused
- Do not hallucinate experience or technologies
- Keep suggestions concise, practical, and actionable
- Focus on ATS-friendly and measurable improvements
- Use proper markdown formatting with headings and bullet points
- Highlight both strengths and weaknesses clearly
- Maintain a professional and analytical tone
"""

In [42]:
user_prompt = f"""
Candidate Profile:
{candidate_info}

Job Description:
{job_description}

Generate a professional markdown report with the following sections:

# Match Summary
- Overall candidate fit
- Recruiter-style evaluation

# Strengths
- Key strengths aligned with the role

# Skill Gaps
- Missing or weak areas

# ATS Improvements
- Missing keywords
- ATS-friendly improvements
- Resume readability suggestions

# Resume Enhancement Ideas
- Better achievement wording
- Quantifiable impact suggestions
- Project improvement ideas

# Interview Preparation
## Technical Topics
## Likely Interview Questions
## Areas to Focus

# Match Score
- Score out of 100
- Short reasoning for the score

Rules:
- Keep the response professional and concise
- Do not hallucinate technologies or experience
- Focus on actionable improvements
- Use proper markdown formatting

"""

### Final step: Give extracted candidate json (for better accuracy) and job description to the LLM and final output in markdown.

In [ ]:
def generate_resume_match_summary():
    response = openai.chat.completions.create(
        model="openchat:7b",
        messages=[
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": user_prompt
            }
        ]
    )

    result = response.choices[0].message.content
    display(Markdown(result))

In [47]:
generate_resume_match_summary()

 # Match Summary

Overall, the candidate exhibits a strong fit for the role of Backend Developer with 2+ years of experience. Their experience in Node.js, Express.js, and MongoDB aligns with the requirements mentioned in the job description. However, they are missing some critical skills such as AWS exposure and knowledge in AI/LLM integrations or automation workflows.

# Strengths
- Proficiency in Node.js, Express.js, and RESTful APIs
- Familiarity with MongoDB and SQL databases
- Exposure to Git and collaborative development workflows
- Basic knowledge of frontend technologies like React.js
- Experience in design, develop, and maintain scalable backend services
- Knowledge of authentication systems such as JWT and OAuth

# Skill Gaps
- No exposure to AI/LLM integrations or automation workflows
- Limited knowledge of cloud platforms such as AWS or Azure (intermediate exposure)
- Lacks experience with TypeScript, Redis, RabbitMQ, Kafka, serverless services, and Kubernetes
- Inadequate understanding of system design, scalable architecture, and secure coding practices

# ATS Improvements
- Add more instances showcasing experience in Node.js, Express.js, MongoDB, and RESTful APIs
- Improve the use of relevant keywords (e.g., Node.js, AWS, Agile/Scrum, serverless services) for better ATS recognition

# Resume Enhancement Ideas
## Better Achievement Wordings
1. "Improved API response times" - Make it quantifiable: "Reduced API response time by up to 30%"
2. "Optimized MongoDB and MySQL databases" - Include more specifics: "Implemented DB sharding and optimization techniques, resulting in a 25% faster query performance and improved data integrity"

## Quantifiable Impact Suggestions
- Calculate exact percentage improvements, number of features developed, or user sessions handled for the achievements mentioned

## Project Improvement Ideas
1. Add projects to showcase AI/LLM integrations or automation workflows
2. Highlight any open-source contributions or personal backend projects to display expertise in the field
3. Show experience building SaaS products or internal developer platforms if applicable

# Interview Preparation
## Technical Topics
1. Deepen understanding of Node.js, Express.js, and RESTful APIs
2. Familiarize with TypeScript, Redis, RabbitMQ, Kafka, serverless services, and Kubernetes
3. Gain insight into database optimization and application scalability
4. Strengthen knowledge in system design and security best practices
## Likely Interview Questions
1. Can you explain how to implement JWT and OAuth authentication mechanisms?
2. How do you handle caching, queues, or background jobs within a Node.js application?
3. What are your strategies for ensuring optimal performance and maintaining scalability in backend systems using Node.js and Express.js?
## Areas to Focus
1. Solidify AI/LLM integration and automation workflows understanding
2. Strengthen exposure to cloud platforms, DevOps, Docker containers, and CI/CD pipelines
3. Gain experience with Agile/Scrum methodologies if not already familiar